# LC 207 — Course Schedule
**Day 54 | Mixed Review Sprint | Difficulty: Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Model prerequisites as a directed
graph. A valid schedule exists if and only if the graph
has no cycle. Detect cycles with DFS tri-colour marking:
0=unvisited, 1=in current path (visiting), 2=fully done.
If DFS ever reaches a node marked 1, a cycle exists.
</div>

## Official Problem Statement

There are a total of `numCourses` courses you have to
take, labelled from `0` to `numCourses - 1`.

You are given an array `prerequisites` where
`prerequisites[i] = [ai, bi]` indicates that you **must**
take course `bi` first if you want to take course `ai`.

Return `true` if you can finish all courses. Otherwise,
return `false`.

**Constraints:**
- `1 <= numCourses <= 2000`
- `0 <= prerequisites.length <= 5000`
- `prerequisites[i].length == 2`
- `0 <= ai, bi < numCourses`
- All the pairs `prerequisites[i]` are **unique**.

## What This Is Actually Asking

Build a directed graph where an edge `b -> a` means
"take b before a". Can you order all courses without
a contradiction? A contradiction is a cycle: A needs B
needs A. This is exactly the problem of checking whether
a directed graph is a DAG (Directed Acyclic Graph).
DFS with state tracking is the standard approach, and
it runs in O(V+E) time.

## Walk Through an Example by Hand

```
numCourses=4
prerequisites=[[1,0],[2,1],[3,2]]   (chain: 0->1->2->3)

Graph (who must come first):
  0 -> [1]   1 -> [2]   2 -> [3]   3 -> []

DFS from 0:
  visit 0: state[0]=1
    visit 1: state[1]=1
      visit 2: state[2]=1
        visit 3: state[3]=1
          no neighbours
        state[3]=2  (done)
      state[2]=2
    state[1]=2
  state[0]=2

No node with state=1 was revisited -> no cycle -> True

---
numCourses=2   prerequisites=[[0,1],[1,0]]   (cycle)

DFS from 0:
  visit 0: state[0]=1
    visit 1: state[1]=1
      neighbour 0: state[0]==1  CYCLE!
      return False
```

## The Picture

```
Tri-colour DFS cycle detection:

  State 0 (WHITE):  never visited
  State 1 (GRAY):   on current DFS stack  <- danger zone
  State 2 (BLACK):  fully explored, safe

No cycle example:          Cycle example:

  0 -> 1 -> 3              0 -> 1
  |         |              ^    |
  v         v              |    v
  2 ------> 4              3 <- 2

DFS(0):                  DFS(0):
  mark 0=GRAY              mark 0=GRAY
  DFS(1):                  DFS(1):
    mark 1=GRAY              mark 1=GRAY
    DFS(3):                  DFS(2):
      mark 3=GRAY              mark 2=GRAY
      DFS(4):                  DFS(3):
        mark 4=GRAY              mark 3=GRAY
        no neighbours            DFS(0):
        mark 4=BLACK               state[0]==GRAY!
      mark 3=BLACK               CYCLE FOUND -> False
    mark 1=BLACK
  DFS(2) -> DFS(4): BLACK, skip
  mark 0=BLACK
  -> True (no cycle)
```

## When To Use This Pattern

- When tasks have dependencies, think **directed graph +
  topological sort or cycle detection**.
- When asked "can X be done without contradiction?",
  think **DAG check via DFS**.
- When you need to know if a path revisits itself during
  DFS, think **tri-colour state array**.
- When nodes can be visited from multiple entry points,
  think **iterate over all nodes** as DFS roots.
- When both cycle detection and ordering are needed,
  think **Kahn's algorithm (BFS + in-degree)**.

## The Approach

Build an adjacency list from the prerequisites. Maintain
a `state` array initialised to 0 for all courses. For
each unvisited course, run DFS: mark it 1 (visiting),
recurse on all its neighbours — if a neighbour is
currently marked 1, a cycle exists, return False. After
exploring all neighbours, mark the node 2 (done). If
all DFS calls complete without finding a cycle, return
True.

In [ ]:
from typing import List
from collections import defaultdict, deque

In [ ]:
def test_harness(func):
    cases = [
        # (numCourses, prerequisites, expected)
        (2, [[1, 0]],             True),
        (2, [[1, 0], [0, 1]],     False),
        (1, [],                   True),
        (4, [[1,0],[2,1],[3,2]],  True),
        (3, [[0,1],[1,2],[2,0]],  False),
        (5, [[1,0],[2,0],[3,1],
             [4,2]], True),
    ]
    passed = 0
    for n, prereqs, expected in cases:
        result = func(n, prereqs)
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        else:
            print(
                f"{status} | n={n} prereqs={prereqs} |"
                f" got={result} expected={expected}"
            )
    print(f"\nResults: {passed}/{len(cases)} passed")
    if passed == len(cases):
        print("All tests PASSED!")

In [ ]:
def can_finish(
    numCourses: int,
    prerequisites: List[List[int]]
) -> bool:
    """
    Return True if all courses can be finished (no cycle).

    Strategy: DFS tri-colour cycle detection.
      state[v] = 0 unvisited
      state[v] = 1 visiting (on current DFS stack)
      state[v] = 2 done (fully explored)
      Cycle detected when we reach a node with state==1.

    Args:
        numCourses:    total number of courses
        prerequisites: list of [a, b] meaning b before a

    Returns:
        bool: True if schedule is possible
    """
    print(
        f"[DEBUG] numCourses={numCourses} "
        f"edges={len(prerequisites)}"
    )

    # Build adjacency list
    graph = defaultdict(list)
    for a, b in prerequisites:
        graph[b].append(a)  # b must come before a

    print(f"[DEBUG] graph={dict(graph)}")

    state = [0] * numCourses  # 0=unvisited,1=visiting,2=done

    def dfs(node):
        if state[node] == 1:
            print(f"[DEBUG] cycle at node={node}")
            return False  # cycle!
        if state[node] == 2:
            return True   # already verified safe
        state[node] = 1
        for neighbour in graph[node]:
            if not dfs(neighbour):
                return False
        state[node] = 2
        return True

    for course in range(numCourses):
        if state[course] == 0:
            if not dfs(course):
                return False

    print("[DEBUG] no cycle found")
    return True


pass

In [ ]:
# Uncomment and run when solution is ready
# test_harness(can_finish)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute Force (try all orderings) | O(V!) | O(V) | Factorial, unusable |
| DFS tri-colour | O(V+E) | O(V+E) | V=courses, E=prereqs |
| Kahn's BFS (in-degree) | O(V+E) | O(V+E) | Also finds the order |

## Real World Connection

At **Citi**, regulatory compliance workflows define task
dependencies (e.g., reconciliation before settlement);
a circular dependency would deadlock the pipeline,
exactly what this algorithm catches. **AWS Step
Functions** validates DAG-shaped workflows before
execution using topological checks. In **data
engineering**, dbt model dependency graphs are DAGs;
circular refs break the build and must be detected
early. Understanding cycle detection is essential for
any orchestration or dependency-management system.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra